In [1]:
# 0 setup

!pip install -q transformers scikit-learn pandas numpy joblib

In [2]:
# 0 imports

import os
import zipfile
import joblib
import pandas as pd
import numpy as np

from transformers import pipeline

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

In [3]:
# 0 load data

zip_path = "/content/archive (8).zip"
extract_folder = "/content/olist"

os.makedirs(extract_folder, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_folder)

orders = pd.read_csv(f"{extract_folder}/olist_orders_dataset.csv")
items = pd.read_csv(f"{extract_folder}/olist_order_items_dataset.csv")
payments = pd.read_csv(f"{extract_folder}/olist_order_payments_dataset.csv")
reviews = pd.read_csv(f"{extract_folder}/olist_order_reviews_dataset.csv")

print("orders:", orders.shape)
print("items:", items.shape)
print("payments:", payments.shape)
print("reviews:", reviews.shape)

orders: (99441, 8)
items: (112650, 7)
payments: (103886, 5)
reviews: (99224, 7)


In [4]:
# 0 rebuild hw2 dataset

orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"], errors="coerce")
orders["order_delivered_customer_date"] = pd.to_datetime(orders["order_delivered_customer_date"], errors="coerce")
orders["order_estimated_delivery_date"] = pd.to_datetime(orders["order_estimated_delivery_date"], errors="coerce")

items_agg = items.groupby("order_id").agg(
    n_items=("order_item_id", "max"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum"),
    mean_price=("price", "mean")
).reset_index()

payments_agg = payments.groupby("order_id").agg(
    payment_value_sum=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", lambda x: x.mode().iloc[0] if len(x.mode()) else np.nan)
).reset_index()

df = orders.merge(
    reviews[["order_id", "review_score", "review_comment_message"]],
    on="order_id",
    how="inner"
)

df = df.merge(items_agg, on="order_id", how="left")
df = df.merge(payments_agg, on="order_id", how="left")

df["is_positive_review"] = (df["review_score"] >= 4).astype(int)

df["delivery_days"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["delivery_vs_estimated"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

df["order_weekday"] = df["order_purchase_timestamp"].dt.weekday
df["order_month"] = df["order_purchase_timestamp"].dt.month

df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_message,...,total_freight,mean_price,payment_value_sum,payment_installments,payment_type,is_positive_review,delivery_days,delivery_vs_estimated,order_weekday,order_month
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,4,"Não testei o produto ainda, mas ele veio corre...",...,8.72,29.99,38.71,1.0,voucher,1,8.0,-8.0,0,10
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,4,Muito bom o produto.,...,22.76,118.70,141.46,1.0,boleto,1,13.0,-6.0,1,7
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,5,NaN,...,19.22,159.90,179.12,3.0,credit_card,1,9.0,-18.0,2,8
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,5,O produto foi exatamente o que eu esperava e e...,...,27.20,45.00,72.20,1.0,credit_card,1,13.0,-13.0,5,11
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,5,NaN,...,8.72,19.90,28.62,1.0,credit_card,1,2.0,-10.0,1,2


In [5]:
# 0 train hw2 model

numeric_features = [
    "delivery_days",
    "delivery_vs_estimated",
    "total_price",
    "total_freight",
    "n_items",
    "payment_value_sum",
    "payment_installments",
    "mean_price",
    "order_weekday",
    "order_month"
]

categorical_features = ["payment_type"]

feature_cols = numeric_features + categorical_features

X = df[feature_cols].copy()
y = df["is_positive_review"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer([
    ("num", num_pipe, numeric_features),
    ("cat", cat_pipe, categorical_features)
])

model = Pipeline([
    ("preprocess", preprocess),
    ("model", HistGradientBoostingClassifier(
        learning_rate=0.1,
        max_depth=5,
        max_iter=200,
        random_state=42
    ))
])

model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:, 1]

hw2_metrics = {
    "accuracy": accuracy_score(y_test, pred),
    "precision": precision_score(y_test, pred),
    "recall": recall_score(y_test, pred),
    "f1": f1_score(y_test, pred),
    "roc_auc": roc_auc_score(y_test, prob)
}

hw2_metrics

{'accuracy': 0.8226253464348703,
 'precision': 0.8249613601236476,
 'recall': 0.9771805936968746,
 'f1': 0.8946423226578869,
 'roc_auc': np.float64(0.7256807013215553)}

In [6]:
# Part 1A setup and inference

df_text = df[df["review_comment_message"].notna()].copy()
df_text = df_text[df_text["review_comment_message"].str.strip() != ""]

sample_df = df_text.sample(n=500, random_state=42).copy()

sample_df["actual"] = (sample_df["review_score"] >= 4).astype(int)

sentiment = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment"
)

def convert_label(label):
    stars = int(label[0])
    return 1 if stars >= 4 else 0

foundation_preds = []

for text in sample_df["review_comment_message"]:
    result = sentiment(str(text)[:512])[0]
    foundation_preds.append(convert_label(result["label"]))

sample_df["foundation_pred"] = foundation_preds

print("Actual distribution:")
print(sample_df["actual"].value_counts())

print("\nFoundation model predicted distribution:")
print(sample_df["foundation_pred"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Actual distribution:
actual
1    321
0    179
Name: count, dtype: int64

Foundation model predicted distribution:
foundation_pred
1    257
0    243
Name: count, dtype: int64


In [7]:
# Part 1B performance comparison

y_true = sample_df["actual"]
y_pred_foundation = sample_df["foundation_pred"]

X_sample = sample_df[feature_cols].copy()
y_pred_hw2 = model.predict(X_sample)

comparison = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Foundation Model": [
        accuracy_score(y_true, y_pred_foundation),
        precision_score(y_true, y_pred_foundation),
        recall_score(y_true, y_pred_foundation),
        f1_score(y_true, y_pred_foundation)
    ],
    "HW2 Model": [
        accuracy_score(y_true, y_pred_hw2),
        precision_score(y_true, y_pred_hw2),
        recall_score(y_true, y_pred_hw2),
        f1_score(y_true, y_pred_hw2)
    ]
})

comparison

,Metric,Foundation Model,HW2 Model
0,Accuracy,0.832000,0.752000
1,Precision,0.961089,0.728538
2,Recall,0.769470,0.978193
3,F1 Score,0.854671,0.835106


# 1C Reflection

On the 500-record sample, the foundation model performed better overall than my HW2 model. The foundation model had higher accuracy, precision, and F1 score. This makes sense because it analyzes the review text directly, which already contains the customer’s opinion. The HW2 model only uses order and delivery features, so it is trying to predict satisfaction before the review is written.

One advantage of the foundation model is that it requires zero training on the Olist dataset. This makes it useful when a company wants a quick solution or does not have time to train a custom model. It also had very strong precision, meaning when it predicted a positive review, it was usually correct. However, the disadvantage is that it is reactive. It can only be used after the customer writes the review, so it cannot help Olist prevent dissatisfaction ahead of time.

The HW2 model had higher recall, which means it was better at catching more positive reviews. This shows that a custom model can still be useful, especially when the goal is early prediction before review text exists.

In production, I would use both models together. The HW2 model could be used as an early-warning system before or around delivery time, while the foundation model could analyze review comments after they are submitted to understand customer feedback more clearly.

In [8]:
# Part 2A install and imports

!pip install -q mlflow

import mlflow
import mlflow.sklearn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.2/866.2 kB 38.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20

In [10]:
# import missing model

from sklearn.ensemble import RandomForestClassifier

In [11]:
# Part 2A logging experiments

mlflow.set_experiment("olist-satisfaction")

models = {
    "HistGradientBoosting": HistGradientBoostingClassifier(max_depth=5, max_iter=200, random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
}

for name, clf in models.items():

    pipe = Pipeline([
        ("preprocess", preprocess),
        ("model", clf)
    ])

    pipe.fit(X_train, y_train)

    preds = pipe.predict(X_test)
    probs = pipe.predict_proba(X_test)[:, 1]

    with mlflow.start_run(run_name=name):

        mlflow.log_param("model_type", name)

        mlflow.log_metrics({
            "accuracy": accuracy_score(y_test, preds),
            "precision": precision_score(y_test, preds),
            "recall": recall_score(y_test, preds),
            "f1": f1_score(y_test, preds),
            "roc_auc": roc_auc_score(y_test, probs)
        })

        mlflow.sklearn.log_model(pipe, "model")

        print(name, "logged")

2026/04/28 03:38:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 03:38:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


HistGradientBoosting logged


2026/04/28 03:39:51 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/28 03:39:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


RandomForest logged


In [12]:
# Part 2B register best model

from mlflow.tracking import MlflowClient
import mlflow

client = MlflowClient()

experiment = client.get_experiment_by_name("olist-satisfaction")

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.f1 DESC"]
)

best_run = runs[0]
run_id = best_run.info.run_id

print("Best run:", best_run.data.tags.get("mlflow.runName"))
print("F1:", best_run.data.metrics["f1"])

model_uri = f"runs:/{run_id}/model"

registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="olist-satisfaction-model"
)

print("Registered model:", registered_model.name)
print("Version:", registered_model.version)

Best run: HistGradientBoosting
F1: 0.8946423226578869


Successfully registered model 'olist-satisfaction-model'.
2026/04/28 03:42:32 WARNING mlflow.tracking._model_registry.fluent: Run with id 5662b39a348d47a5b28b75c7a8fe30eb has no artifacts at artifact path 'model', registering model based on models:/m-9516023562c14adea9749f54e7f9b31b instead


Registered model: olist-satisfaction-model
Version: 1


Created version '1' of model 'olist-satisfaction-model'.


In [13]:
# Part 2B move model to production

client.transition_model_version_stage(
    name="olist-satisfaction-model",
    version=registered_model.version,
    stage="Production"
)

print("Model moved to Production")

Model moved to Production


/tmp/ipykernel_2578/356322477.py:3: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


#2B Screenshots

On PDF

In [29]:
# Part 3A save model and metadata

import os
import json
import joblib

PROJECT_DIR = "/content/hw4-mlops"
MODEL_DIR = f"{PROJECT_DIR}/model"

os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(model, f"{MODEL_DIR}/model.pkl")

metadata = {
    "feature_cols": feature_cols,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "allowed_payment_types": sorted(df["payment_type"].dropna().unique().tolist())
}

with open(f"{MODEL_DIR}/metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

print("model and metadata saved")

model and metadata saved


In [30]:
# Part 3A create flask app

app_code = '''
from flask import Flask, request, jsonify
import joblib
import json
import pandas as pd

app = Flask(__name__)

model = joblib.load("model/model.pkl")

with open("model/metadata.json", "r") as f:
    metadata = json.load(f)

required_features = metadata["feature_cols"]
numeric_features = metadata["numeric_features"]
categorical_features = metadata["categorical_features"]
allowed_payment_types = metadata["allowed_payment_types"]

def validate_input(data):
    errors = {}

    for field in required_features:
        if field not in data:
            errors[field] = "missing required field"

    if errors:
        return errors

    for field in numeric_features:
        value = data[field]

        if not isinstance(value, (int, float)):
            errors[field] = "must be a number"

        if field in ["delivery_days", "total_price", "total_freight", "n_items", "payment_value_sum", "payment_installments", "mean_price"]:

            if isinstance(value, (int, float)) and value < 0:
                errors[field] = "must be non-negative"

    if "order_weekday" in data:
        if data["order_weekday"] < 0 or data["order_weekday"] > 6:
            errors["order_weekday"] = "must be between 0 and 6"

    if "order_month" in data:
        if data["order_month"] < 1 or data["order_month"] > 12:
            errors["order_month"] = "must be between 1 and 12"

    if "payment_type" in data:
        if data["payment_type"] not in allowed_payment_types:
            errors["payment_type"] = f"invalid value. allowed values are {allowed_payment_types}"

    return errors

@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "healthy",
        "model": "loaded"
    })

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    if not isinstance(data, dict):
        return jsonify({"error": "Invalid input", "details": "Input must be a JSON object"}), 400

    errors = validate_input(data)

    if errors:
        return jsonify({"error": "Invalid input", "details": errors}), 400

    X = pd.DataFrame([data], columns=required_features)

    prediction = int(model.predict(X)[0])
    probability = float(model.predict_proba(X)[0][1])

    label = "positive" if prediction == 1 else "negative"

    return jsonify({
        "prediction": prediction,
        "probability": probability,
        "label": label
    })

@app.route("/predict/batch", methods=["POST"])
def predict_batch():
    data = request.get_json()

    if not isinstance(data, list):
        return jsonify({"error": "Invalid input", "details": "Input must be a list"}), 400

    if len(data) > 100:
        return jsonify({"error": "Invalid input", "details": "Maximum batch size is 100"}), 400

    results = []

    for record in data:
        errors = validate_input(record)

        if errors:
            return jsonify({"error": "Invalid input", "details": errors}), 400

        X = pd.DataFrame([record], columns=required_features)

        prediction = int(model.predict(X)[0])
        probability = float(model.predict_proba(X)[0][1])

        label = "positive" if prediction == 1 else "negative"

        results.append({
            "prediction": prediction,
            "probability": probability,
            "label": label
        })

    return jsonify(results)

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''

with open("/content/hw4-mlops/app.py", "w") as f:
    f.write(app_code)

print("app.py created")

app.py created


In [31]:
# Part 3B create test api file

test_code = '''
import requests

BASE_URL = "http://127.0.0.1:5000"

valid_record = {
    "delivery_days": 12,
    "delivery_vs_estimated": 3,
    "total_price": 149.99,
    "total_freight": 25.50,
    "n_items": 1,
    "payment_value_sum": 175.49,
    "payment_installments": 3,
    "mean_price": 149.99,
    "order_weekday": 2,
    "order_month": 5,
    "payment_type": "credit_card"
}

print("Test 1: Health check")
r = requests.get(f"{BASE_URL}/health")
print(r.status_code, r.json())

print("Test 2: Single prediction")
r = requests.post(f"{BASE_URL}/predict", json=valid_record)
print(r.status_code, r.json())

print("Test 3: Batch prediction")
batch = [valid_record.copy() for i in range(5)]
r = requests.post(f"{BASE_URL}/predict/batch", json=batch)
print(r.status_code, r.json())
print("Number of predictions:", len(r.json()))

print("Test 4: Missing required field")
bad_record = valid_record.copy()
bad_record.pop("delivery_days")
r = requests.post(f"{BASE_URL}/predict", json=bad_record)
print(r.status_code, r.json())

print("Test 5: Invalid type")
bad_record = valid_record.copy()
bad_record["total_price"] = "expensive"
r = requests.post(f"{BASE_URL}/predict", json=bad_record)
print(r.status_code, r.json())

print("All tests completed")
'''

with open("/content/hw4-mlops/test_api.py", "w") as f:
    f.write(test_code)

print("test_api.py created")

test_api.py created


Public Url: https://hw4-mlops-68aw.onrender.com/

GitHub Repisotory Link: https://github.com/kevc10/hw4-mlops.git